# 06_validations - TO BE DISCUSSED LATER

Intentionally empty notebook: we will decide its content together. Tests considered so far (updated for the current plan):

1. **F1 out of sample**: train EC-Earth3 historical 1981-2003 (step1 with `TRAIN_PERIOD`), apply to 2004-2014, compare with ERA5 2004-2014.
   There is no day-by-day consistency, so compare **distributions** (quantiles 50 / 95 / 99, dry-day fraction, drizzle, seasonal cycle, maps), never day-by-day series.
2. **F1 in a warmer climate**: analogues are searched in the historical climate. For the 2025 / 2050 runs (especially hosing), check the share of days / pixels whose values fall outside the historical range (mainly `tasmax`).
3. **GCM signal preservation** by F1: 2025 - historical, and hosing - no-hosing, before / after (notebook step2 gives a first look).
4. **Residual bias** of the downscaled GCM against ERA5 over the historical period: if a per-pixel marginal bias remains, consider an additional WDF + QDM (decision to take after this test).
5. **F2 on two halves** of the RCM 2025 no-hosing run (25 / 25 years).
6. **Hosing delta before / after F2** + **range test**: per pixel and per month, bounds of the raw RCM no-hosing run (min/max or 0.5 / 99.5 percentiles), fraction of hosing days outside the bounds (upper bound only for `pr`), with the second half vs the first half of the no-hosing run as the baseline.
7. **Indicative comparison with ISIMIP3b EC-Earth3 (0.5 deg)**: sanity check only (W5E5 is not ERA5).

In [ ]:
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']
BBOX = {"lat": tuple(cfg["europe_bbox"]["lat"]), "lon": tuple(cfg["europe_bbox"]["lon"])}

# TODO: to be written after discussion